In [1]:
# Homework 10
# 2 Units with Astropy
# 2.1 Clean the Data
# 2.1.1 Fix Formatting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

In [2]:
df = pd.read_csv("solar_system.csv")
print(df.shape)

(20, 11)


In [3]:
df2 = df.set_index("Attribute").T
print(df2.shape)
df2.index.name = "Planet"
print(df2.shape)
df2.reset_index(inplace=True)
print(df2.shape)
df2.columns.name = None
print(df2.shape)

(10, 20)
(10, 20)
(10, 21)
(10, 21)


In [4]:
# b) There are originally 20 rows and 11 columns, and after transposing there are 10 rows and 21 columns
''' c) The first line sets the attribute column as the index and switches the rows and the columns.
The second line sets labels the index column as planets.
The third line turns the planet index into an actual column.
The fourth line removes the name for the column index. '''
# d) It isn't simply flipped because we added a column labeled planet, and merged the attribute column with the index. So there is one more column and one less row than there would be otherwise.
print(df2.columns)
# f) 16 columns have units. They are mass, diameter, density, gravity, escape velocity, rotation period, length of day, distance from sun, perihelion, aphelion, orbital period, orbital velocity, orbital inclination, obliquity to orbit, mean tempreature, and surface pressure.
# 5 columns don't have units. They are planet, orbital eccentricity, number of moons, ring system, and global magnetic field.
print(df2)

Index(['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object')
    Planet Mass (10^24kg) Diameter (km) Density (kg/m^3) Gravity (m/s^2)  \
0  Mercury          0.330          4879             5429             3.7   
1    Venus           4.87         12104             5243             8.9   
2    Earth           5.97         12756             5514             9.8   
3     Moon          0.073          3475             3340             1.6   
4     Mars          0.642          6792             3934 

In [5]:
# 2.1.2 Investigating Data Types
print(df.dtypes)

Attribute    object
Mercury      object
Venus        object
Earth        object
Moon         object
Mars         object
Jupiter      object
Saturn       object
Uranus       object
Neptune      object
Pluto        object
dtype: object


In [6]:
for col in df2.columns:
    print(df2[col].apply(type).value_counts())
    print()

Planet
<class 'str'>    10
Name: count, dtype: int64

Mass (10^24kg)
<class 'str'>    10
Name: count, dtype: int64

Diameter (km)
<class 'str'>    10
Name: count, dtype: int64

Density (kg/m^3)
<class 'str'>    10
Name: count, dtype: int64

Gravity (m/s^2)
<class 'str'>    10
Name: count, dtype: int64

Escape Velocity (km/s)
<class 'str'>    10
Name: count, dtype: int64

Rotation Period (hours)
<class 'str'>    10
Name: count, dtype: int64

Length of Day (hours)
<class 'str'>    10
Name: count, dtype: int64

Distance from Sun (10^6 km)
<class 'str'>    10
Name: count, dtype: int64

Perihelion (10^6 km)
<class 'str'>    10
Name: count, dtype: int64

Aphelion (10^6 km)
<class 'str'>    10
Name: count, dtype: int64

Orbital Period (days)
<class 'str'>    10
Name: count, dtype: int64

Orbital Velocity (km/s)
<class 'str'>    10
Name: count, dtype: int64

Orbital Inclination (deg)
<class 'str'>    10
Name: count, dtype: int64

Orbital Eccentricity
<class 'str'>    10
Name: count, dtype: int

In [7]:
# a) the actual values are strings
# b) this could be a problem because it is hard to do numeric operations on strings in pandas

In [8]:
# 2.1.3 Convert Strings to Numbers
for col in df2.columns:
    if col not in ["Planet", "Ring System?", "Global Magnetic Field?"]:
        df2[col] = pd.to_numeric(df2[col], errors = "coerce")

In [9]:
print(df2.dtypes)

Planet                          object
Mass (10^24kg)                 float64
Diameter (km)                    int64
Density (kg/m^3)                 int64
Gravity (m/s^2)                float64
Escape Velocity (km/s)         float64
Rotation Period (hours)        float64
Length of Day (hours)          float64
Distance from Sun (10^6 km)    float64
Perihelion (10^6 km)           float64
Aphelion (10^6 km)             float64
Orbital Period (days)          float64
Orbital Velocity (km/s)        float64
Orbital Inclination (deg)      float64
Orbital Eccentricity           float64
Obliquity to Orbit (deg)       float64
Mean Temperature (C)             int64
Surface Pressure (bars)        float64
Number of Moons                  int64
Ring System?                    object
Global Magnetic Field?          object
dtype: object


In [10]:
# a) float64, int64, and object
# b) float64: decimal number, int64: whole number, object: in this case the objects are strings
# c) Global magnetic field, ring system, and planet are all still objects because they should be strings as they are not numeric values

In [11]:
# 2.2 Apply Astropy Units
def attach_units(old_col, new_col, unit):
    df2[old_col] = [val * unit for val in df2[old_col]]
    df2.rename(columns={old_col: new_col}, inplace=True)
attach_units("Mass (10^24kg)", "Mass (kg)", 1e24 * u.kg)
print(df2["Mass (kg)"])

0                   3.3e+23 kg
1                  4.87e+24 kg
2     5.969999999999999e+24 kg
3                   7.3e+22 kg
4                  6.42e+23 kg
5                 1.898e+27 kg
6                  5.68e+26 kg
7                  8.68e+25 kg
8    1.0199999999999999e+26 kg
9                   1.3e+22 kg
Name: Mass (kg), dtype: object


In [12]:
# 2.3 Data Analysis
# 2.3.1
semi_major_axis = (df2["Perihelion (10^6 km)"] + df2["Aphelion (10^6 km)"]) / 2
index = df2.columns.get_loc("Aphelion (10^6 km)") + 1
df2.insert(index, "Semi-Major Axis (10^6 km)", semi_major_axis)
attach_units("Semi-Major Axis (10^6 km)", "Semi-Major Axis (km)", 1e6 * u.km)
print(df2.columns)
print(df2["Semi-Major Axis (km)"])
print(df2["Orbital Period (days)"])


Index(['Planet', 'Mass (kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Semi-Major Axis (km)',
       'Orbital Period (days)', 'Orbital Velocity (km/s)',
       'Orbital Inclination (deg)', 'Orbital Eccentricity',
       'Obliquity to Orbit (deg)', 'Mean Temperature (C)',
       'Surface Pressure (bars)', 'Number of Moons', 'Ring System?',
       'Global Magnetic Field?'],
      dtype='object')
0      57900000.0 km
1     108200000.0 km
2     149600000.0 km
3        384500.0 km
4     228000000.0 km
5     778500000.0 km
6    1432050000.0 km
7    2867050000.0 km
8    4515000000.0 km
9    5906350000.0 km
Name: Semi-Major Axis (km), dtype: object
0       88.0
1      224.7
2      365.2
3       27.3
4      687.0
5     4331.0
6    10747.0
7    30589.0
8    59800.0
9    90560.0
Name: Orbital Period (days), 

In [13]:
# 2.3.2
op_days = df2["Orbital Period (days)"].values
op_years = (op_days * u.day).to(u.year).value
df2["Orbital Period (years)"] = op_years
print(df2.columns)
print(df2["Orbital Period (years)"])
saturn_period = df2.loc[df2["Planet"] == "Saturn", "Orbital Period (years)"].iloc[0]
print(f"Saturn's orbital period: {saturn_period:.4f} years")

Index(['Planet', 'Mass (kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Semi-Major Axis (km)',
       'Orbital Period (days)', 'Orbital Velocity (km/s)',
       'Orbital Inclination (deg)', 'Orbital Eccentricity',
       'Obliquity to Orbit (deg)', 'Mean Temperature (C)',
       'Surface Pressure (bars)', 'Number of Moons', 'Ring System?',
       'Global Magnetic Field?', 'Orbital Period (years)'],
      dtype='object')
0      0.240931
1      0.615195
2      0.999863
3      0.074743
4      1.880903
5     11.857632
6     29.423682
7     83.748118
8    163.723477
9    247.939767
Name: Orbital Period (years), dtype: float64
Saturn's orbital period: 29.4237 years


In [14]:
# 2.3.3
# 1 AU is the distance between the Earth and the Sun
print(const.au)
print(const.au.to(u.km))
# There are 149597870.7 km in an AU

cols_km = ["Diameter (km)", "Distance from Sun (10^6 km)", "Perihelion (10^6 km)", "Aphelion (10^6 km)"]

for col in cols_km:
    km_values = df2[col].values
    if "10^6" in col:
        km_values = km_values * 1e6
    au_values = (km_values * u.km).to(u.AU).value
    new_col = col.replace("(km)", "(AU)").replace("(10^6 km)", "(AU)")
    df2[new_col] = au_values

for col in df2.columns:
    if "(AU)" in col:
        value = df2.loc[df2["Planet"] == "Saturn", col].iloc[0]
        print(f"{col} for {"Saturn"}: {value}")

  Name   = Astronomical Unit
  Value  = 149597870700.0
  Uncertainty  = 0.0
  Unit  = m
  Reference = IAU 2012 Resolution B2
149597870.70000002 km
Diameter (AU) for Saturn: 0.0008057333933697493
Distance from Sun (AU) for Saturn: 9.572328759088414
Perihelion (AU) for Saturn: 9.074995477191642
Aphelion (AU) for Saturn: 10.070330499697413


In [15]:
df2.to_csv("units.csv", index=False)